In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers, ops
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
#import cv2

import os
import sys
sys.path.append('../../utils')
from  configloader import Configloader
import cnn_helpers
from vit_helpers import ImageDataGeneratorFromDF, PatchEncoder, Patches


ModuleNotFoundError: No module named 'tensorflow.python.trackable'

In [4]:
tf.version

AttributeError: module 'tensorflow' has no attribute 'version'

In [6]:
SHAPE = 72             #224   #required for resnet
BATCH_SIZE = 128        #256     #how big ar teh batches for the learning part
MAX_EPOCHS = 100        #upper limit of epochs per learning task - 
                        #    NOTE THAT there is early stopping and lr plateau just as in nobteook 2.
CROP = True             #Based on notebook 3.2
PHASE = 'brand phase'

PATCH_SIZE = 6  #6  #28                 #16 is standard practice, start here, SHAPE%PATCH_SIZE should be 0
PATCHES = (SHAPE/PATCH_SIZE)**2    #projected tokens required for transformer


In [7]:
assert SHAPE%PATCH_SIZE == 0, f'Patch size of {PATCH_SIZE} does not fit in Shape Size of {SHAPE}!!'

In [8]:
cnn_helpers.system_override()
device = cnn_helpers.system_pick_device()

System override applied - check if GPU is detected
Using CPU for deep learning.


In [ ]:
config = Configloader()

basedir = config.get('settings', 'image_directory')
augmdir = config.get('dir_augmentations', 'subfolder')
augmcsv = config.get('dir_augmentations', 'csv_dir')



augment_base = os.path.join(basedir, augmdir, PHASE)
augment_csv_dump = os.path.join(basedir, augmcsv, PHASE)

In [ ]:
traindata = pd.read_csv(os.path.join(augment_csv_dump, 'traindata_brandphase.csv'))
testdata = pd.read_csv(os.path.join(augment_csv_dump, 'testdata_brandphase.csv'))

In [ ]:
angles = traindata['model_label'].unique()


In [ ]:
brands = traindata.brand.unique()
label_encoder = LabelEncoder()
label_encoder.fit(brands)

In [ ]:
#apply label encoding on train and test dataframes. 
traindata['y_encoded'] = label_encoder.transform(traindata['brand'])
testdata['y_encoded'] = label_encoder.transform(testdata['brand'])

In [ ]:
testdata = cnn_helpers.shuffle_df(testdata)
traindata = cnn_helpers.shuffle_df(traindata)

In [ ]:
print(len(traindata))
print(len(testdata))
print(len(brands))

In [ ]:
angles = traindata.model_label.unique()
angles = ['front']  #hack
model_dest_dir = os.path.join(os.getcwd(), '..', '..', 'models', 'ALL_brand_models_angled')
os.makedirs(model_dest_dir, exist_ok=True)

for angle in angles: 
    # Check if angle is trained already: 
    name = f'ViT-model_cropped={CROP}_angle={angle}.keras'
    if name in os.listdir(model_dest_dir):
        print(f"Skipping {name} -- model already trained")
        continue
    # view_by_angle_test = testdata.query('model_label==@angle').reset_index()
    # view_by_angle_train = traindata.query('model_label==@angle').reset_index()
    # X_train, y_train = cnn_helpers.get_X_y(view_by_angle_train, 'y_encoded', ['brand'])
    # X_test, y_test = cnn_helpers.get_X_y(view_by_angle_test, 'y_encoded', ['brand'])


In [ ]:
angle = 'front'
view_by_angle_test = testdata.query('model_label==@angle').reset_index()
view_by_angle_train = traindata.query('model_label==@angle').reset_index()

In [ ]:
view_by_angle_train

In [ ]:
train_generator = ImageDataGeneratorFromDF(df=view_by_angle_train, 
                                              image_column='abs_path', 
                                              label_column='y_encoded', 
                                              batch_size=BATCH_SIZE, 
                                              target_size=(SHAPE, SHAPE), 
                                              shuffle=True, 
                                              apply_crop=CROP)

validation_generator = ImageDataGeneratorFromDF(df=view_by_angle_test, 
                                                   image_column='abs_path', 
                                                   label_column='y_encoded', 
                                                   batch_size=BATCH_SIZE, 
                                                   target_size=(SHAPE, SHAPE), 
                                                   shuffle=True, 
                                                   apply_crop=CROP)

In [ ]:

#test the generators: trainview
floored = len(view_by_angle_train) // BATCH_SIZE
images, labels = train_generator[np.random.randint(0,floored)]
instance = np.random.randint(0, len(images))
plt.imshow(images[instance])
plt.title(f"Label: {labels[instance]}") 


In [ ]:
#test the generators: testview
floored = len(view_by_angle_test) // BATCH_SIZE
images, labels = validation_generator[np.random.randint(0,floored)]
instance = np.random.randint(0, len(images))
plt.imshow(images[instance])
plt.title(f"Label: {labels[instance]}") 

In [ ]:
# import numpy as np
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers
# from tensorflow.keras.utils import Sequence
# import matplotlib.pyplot as plt
# from PIL import Image

# Set your variables
num_classes = len(brands)  # Change number of classes to 30 as per your use case
input_shape = (SHAPE, SHAPE, 3)  # Adjust the shape based on your generator's target size

# We assume that your generators are defined as:
# train_generator = ImageDataGeneratorFromDF(df=view_by_angle_train, ...)
# validation_generator = ImageDataGeneratorFromDF(df=view_by_angle_test, ...)

"""
## Configure the hyperparameters
"""

learning_rate = 0.001
weight_decay = 0.0001
batch_size = BATCH_SIZE
num_epochs = MAX_EPOCHS             # For real training, use num_epochs=100. 10 is a test value
image_size = SHAPE          # You can keep this or adjust it as needed
patch_size = PATCH_SIZE              # Size of the patches to be extract from the input images
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
]  # Size of the transformer layers
transformer_layers = 8
mlp_head_units = [
    2048,
    1024,
]  # Size of the dense layers of the final classifier

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=keras.activations.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


def create_vit_classifier():
    inputs = keras.Input(shape=input_shape)
    # Skip data augmentation as per the request
    # augmented = data_augmentation(inputs)
    augmented = inputs
    
    # Create patches.
    patches = Patches(patch_size)(augmented)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    # Classify outputs.
    logits = layers.Dense(num_classes)(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)
    return model


"""
## Compile, train, and evaluate the model using generators
"""


def run_experiment_with_generators(model):
    optimizer = keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(5, name="top-5-accuracy"),
        ],
    )

    checkpoint_filepath = "/tmp/checkpoint.weights.h5"
    checkpoint_callback = keras.callbacks.ModelCheckpoint(
        checkpoint_filepath,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    )

    history = model.fit(
        train_generator,  # Use the generator for training
        epochs=num_epochs,
        validation_data=validation_generator,  # Use the generator for validation
        callbacks=[checkpoint_callback],
    )

    model.load_weights(checkpoint_filepath)
    _, accuracy, top_5_accuracy = model.evaluate(validation_generator)
    print(f"Test accuracy: {round(accuracy * 100, 2)}%")
    print(f"Test top 5 accuracy: {round(top_5_accuracy * 100, 2)}%")

    return history


vit_classifier = create_vit_classifier()
history = run_experiment_with_generators(vit_classifier)


def plot_history(item):
    plt.plot(history.history[item], label=item)
    plt.plot(history.history["val_" + item], label="val_" + item)
    plt.xlabel("Epochs")
    plt.ylabel(item)
    plt.title("Train and Validation {} Over Epochs".format(item), fontsize=14)
    plt.legend()
    plt.grid()
    plt.show()


plot_history("loss")
plot_history("top-5-accuracy")


In [ ]:
len(view_by_angle_test)